# S4 J5 — Workflow Engine

Ce notebook est généré à partir du Markdown source du jour.

## Objectifs

- Comprendre le rôle d'un workflow engine.
- Modéliser un graphe d'étapes.
- Exécuter un workflow déterministe.
- Observer retries, conditions, blocages et traces.

## Modèle mental

Un workflow engine transforme un processus métier en contrat exécutable : étapes, dépendances, statuts, approbations et traces.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
while ROOT.name != "ai-engineering-bootcamp" and ROOT.parent != ROOT:
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from mini_framework.workflow import WorkflowEngine
from book.week04.day05.labs.workflow_engine_lab import build_support_workflow, register_support_handlers

engine = WorkflowEngine()
register_support_handlers(engine)
workflow = build_support_workflow()
engine.manifest(workflow)

## Exécution avec approbation

L'étape sensible `refund` peut s'exécuter lorsque l'approbation humaine est fournie.

In [ ]:
run = engine.run(
    workflow,
    input_payload={
        "message": "Urgent refund request for a duplicated charge",
        "customer_tier": "premium",
        "amount": 42.0,
    },
    approvals={"refund"},
    run_id="notebook_run_approved",
)
run.status, run.step_results["refund"].status, run.state.data["final_answer"]

## Exécution sans approbation

Le moteur bloque l'étape sensible et évite de produire une fausse finalisation.

In [ ]:
blocked_run = engine.run(
    workflow,
    input_payload={"message": "refund please", "amount": 42.0},
    approvals=set(),
    run_id="notebook_run_blocked",
)
blocked_run.status, blocked_run.step_results["refund"].status, blocked_run.step_results["final_answer"].status

## Lire la trace

In [ ]:
[(event.step, event.event, event.payload) for event in blocked_run.trace]

## Exercices

1. Ajoutez une étape sensible `notify_customer` après `final_answer`.
2. Ajoutez un test qui vérifie le blocage sans approbation.
3. Modifiez le workflow pour que `final_answer` puisse produire un résumé même si `refund` est bloqué.
4. Expliquez le risque métier associé à cette modification.

## Solutions formateur

Les corrections complètes sont disponibles dans `book/week04/day05/corriges/`.


# Corrections — Exercises

## Exercice 1

1. déterminer l'ordre des étapes — workflow engine ;
2. appeler une API métier — handler, éventuellement via tool registry ;
3. stocker une préférence utilisateur durable — memory layer ;
4. décider si une étape sensible est approuvée — workflow engine avec politique d'approbation ;
5. produire une trace d'exécution — workflow engine / observability ;
6. valider que le graphe n'a pas de cycle — workflow engine ;
7. générer une réponse naturelle — modèle ou handler de réponse ;
8. vérifier que l'outil existe — tool registry.

## Exercice 2

Workflow possible :

```python
WorkflowDefinition(
    name="lead_qualification",
    version="1.0.0",
    steps=[
        WorkflowStep(name="classify_lead", handler="classify_lead"),
        WorkflowStep(name="enrich_company", handler="enrich_company", depends_on=("classify_lead",)),
        WorkflowStep(name="score_lead", handler="score_lead", depends_on=("enrich_company",)),
        WorkflowStep(
            name="request_sales_review",
            handler="request_sales_review",
            depends_on=("score_lead",),
            condition="score_above_80",
            sensitive=True,
        ),
        WorkflowStep(
            name="final_summary",
            handler="final_summary",
            depends_on=("score_lead", "request_sales_review"),
        ),
    ],
)
```

## Exercice 3

1. une étape a fini sans erreur — `completed` ;
2. une étape attend une validation humaine — `blocked` ;
3. une étape n'a pas été lancée car sa condition est fausse — `skipped` ;
4. une étape a levé une erreur après tous ses retries — `failed` ;
5. le workflow a démarré mais n'est pas terminé — `running`.

## Exercice 4

1. `plan` a utilisé deux tentatives.
2. Le workflow n'a pas échoué : il finit en `completed`.
3. `max_retries` est au moins égal à 1.
4. La trace permet de comprendre qu'il y a eu une erreur temporaire, un retry, puis une récupération. En production, cela aide le debugging, les métriques et l'audit.

## Exercice 5

Sans approbation `refund`, le moteur détecte que l'étape est sensible. Il ne l'exécute pas et retourne `blocked`. L'étape finale dépend de `refund`, donc elle est `skipped`.

## Exercice 6

Modification possible :

```python
WorkflowStep(
    name="notify_customer",
    handler="notify_customer",
    depends_on=("final_answer",),
    sensitive=True,
)
```

Tests à ajouter :

- sans approbation, `notify_customer` est `blocked` ;
- avec approbation, l'étape retourne `{ "sent": True }` ;
- si `final_answer` échoue, `notify_customer` est `skipped` ;
- la trace contient `step.blocked` ou `step.completed` selon le cas.



# Corrections — Interview

## Réponse 1

Un workflow engine rend explicite le processus métier que l'agent doit suivre. Il apporte un contrat d'exécution, des dépendances, des statuts, des retries, des garde-fous et une trace. Cela améliore la testabilité et l'auditabilité.

## Réponse 2

Une boucle agentique est dynamique : le système observe, décide, agit, puis réévalue. Un workflow déterministe est contractuel : les étapes et dépendances sont définies à l'avance. Les deux peuvent coexister : une étape de workflow peut contenir une boucle agentique limitée.

## Réponse 3

On détecte un cycle avec un tri topologique. Si l'algorithme ne peut pas ordonner toutes les étapes, il reste des dépendances non résolues : le graphe contient un cycle.

## Réponse 4

Un prompt est une consigne, pas une garantie d'exécution. Une action sensible doit être contrôlée par le moteur ou une couche de politique, car cette couche est testable, observable et indépendante du comportement probabiliste du modèle.

## Réponse 5

Une trace utile contient au minimum : identifiant de run, nom du workflow, étape, événement, timestamp, tentative, handler, erreur éventuelle, raison de skip ou de blocage, statut final.

## Réponse 6

Il faut rendre les actions idempotentes, tracer les tentatives, limiter les retries, distinguer erreurs temporaires et erreurs définitives, et éviter de réexécuter une action externe non idempotente sans clé d'idempotence.

## Réponse 7

`failed` signifie erreur technique ou métier non récupérée. `blocked` signifie attente d'une approbation ou d'une politique. `skipped` signifie étape volontairement non exécutée, par condition fausse ou dépendance non satisfaite.

## Réponse 8

Pour aller vers la production, il faut ajouter persistance, reprise après crash, exécution asynchrone, queue, timeouts réels, idempotence, annulation, métriques, versioning, secrets, gestion de droits et observabilité centralisée.



# Correction — Challenge

## Implémentation indicative

```python
from mini_framework.workflow import WorkflowDefinition, WorkflowStep, WorkflowEngine

workflow = WorkflowDefinition(
    name="ai_incident_diagnostic",
    version="1.0.0",
    steps=[
        WorkflowStep(name="classify_severity", handler="classify_severity"),
        WorkflowStep(name="collect_context", handler="collect_context", depends_on=("classify_severity",)),
        WorkflowStep(name="hypothesize_root_cause", handler="hypothesize_root_cause", depends_on=("collect_context",)),
        WorkflowStep(
            name="rollback",
            handler="rollback",
            depends_on=("hypothesize_root_cause",),
            condition="should_rollback",
            sensitive=True,
        ),
        WorkflowStep(
            name="final_summary",
            handler="final_summary",
            depends_on=("hypothesize_root_cause", "rollback"),
        ),
    ],
)
```

## Handlers indicatifs

```python
def classify_severity(ctx):
    if ctx.input["error_rate"] > 0.10 or ctx.input["latency_p95_ms"] > 4000:
        return {"severity": "critical"}
    return {"severity": "warning"}

def collect_context(ctx):
    return {
        "service": ctx.input["service"],
        "recent_deploy": ctx.input["recent_deploy"],
        "severity": ctx.data["classify_severity"]["severity"],
    }

def hypothesize_root_cause(ctx):
    if ctx.input["recent_deploy"]:
        cause = "recent deployment regression"
    else:
        cause = "runtime degradation"
    return {"cause": cause}

def rollback(ctx):
    return {"rollback_triggered": True, "service": ctx.input["service"]}

def final_summary(ctx):
    rollback_result = ctx.data.get("rollback")
    return {
        "severity": ctx.data["classify_severity"]["severity"],
        "cause": ctx.data["hypothesize_root_cause"]["cause"],
        "rollback": bool(rollback_result),
    }

def should_rollback(state):
    return (
        state.input.get("recent_deploy") is True
        and state.data["classify_severity"]["severity"] == "critical"
    )
```

## Tests attendus

1. alerte critique sans approbation : `rollback` est `blocked` ;
2. alerte critique avec approbation : `rollback` est `completed` ;
3. alerte non critique : `rollback` est `skipped` ;
4. trace finale contient `workflow.finished` ;
5. la synthèse finale reflète le cas exécuté.

## Point d'attention

Si `final_summary` dépend strictement de `rollback`, elle sera `skipped` lorsque `rollback` est `blocked`. Pour produire une synthèse même en cas de blocage, on peut créer une étape alternative `blocked_summary` ou ajuster le graphe. Dans le lab principal, le choix est conservateur : une action sensible bloquée bloque la suite dépendante.



# Review notes — Jour 5

## Objectif pédagogique atteint

L'étudiant doit repartir avec l'idée qu'un workflow engine est une couche de contrôle, pas une simple fonction utilitaire.

## Points clés à vérifier

- L'étudiant sait lire un graphe de dépendances.
- L'étudiant sait expliquer pourquoi les actions sensibles sont bloquées sans approbation.
- L'étudiant comprend que `skipped` n'est pas une erreur.
- L'étudiant sait lire une trace.
- L'étudiant sait distinguer workflow state et memory layer.

## Erreurs fréquentes

### 1. Confondre retry et boucle infinie

Un retry est borné, tracé et justifié. Une boucle infinie est un défaut d'architecture.

### 2. Rendre l'étape finale trop permissive

Si une action critique échoue ou est bloquée, la réponse finale ne doit pas faire croire que tout est résolu.

### 3. Mettre la sécurité dans le prompt

Le prompt peut expliquer la politique. Le moteur doit l'appliquer.

### 4. Oublier les tests de graphe

Un workflow invalide ne doit jamais démarrer.

## Questions de consolidation

- Où placerais-tu une étape d'évaluation automatique ?
- Comment reprendrais-tu un workflow interrompu ?
- Comment stockerais-tu les traces ?
- Comment éviterais-tu un double remboursement lors d'un retry ?
- Quelles étapes devraient être parallélisées en production ?

## Transition vers le jour 6

Le jour 6 ajoute l'observabilité. Le workflow engine produit déjà des traces ; il faut maintenant structurer ces signaux pour le debugging, le monitoring, l'évaluation et la supervision opérationnelle.
